# Capstone Black Box Project

## Read the data files

# Submission 6 of 13

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
import math

In [62]:
# Define the function being analysed
function = "function_8"

In [63]:
# Load the .npy file

def load_npy_files(function_name):
    initial_inputs = np.load(f'./initial_data/{function_name}/initial_inputs.npy')
    initial_outputs = np.load(f'./initial_data/{function_name}/initial_outputs.npy')
    return initial_inputs, initial_outputs

inputs, outputs = load_npy_files(function)

X=inputs
y=outputs

print(inputs)
print(outputs)
print(y)

[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.41010452
  

In [27]:
dfi = pd.DataFrame(inputs)
print(dfi.describe())
dfo = pd.DataFrame(outputs)
print(dfo.describe())


               0          1          2          3
count  35.000000  35.000000  35.000000  35.000000
mean    0.513012   0.471542   0.473579   0.485207
std     0.288005   0.285970   0.256734   0.283320
min     0.023625   0.006250   0.042186   0.081517
25%     0.266542   0.189025   0.258946   0.246425
50%     0.442669   0.489220   0.453192   0.467193
75%     0.742658   0.696280   0.708006   0.746945
max     0.985622   0.919592   0.948451   0.999483
               0
count  35.000000
mean  -15.785213
std     8.950661
min   -32.625660
25%   -21.048893
50%   -15.487083
75%   -11.632837
max    -0.042435


# Observe data points

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, s=80)
plt.colorbar(label="Function output")
plt.xlabel("Input variable 1")
plt.ylabel("Input variable 2")
plt.title("Observed Function Values")
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(X[:, 0], X[:, 1], y, s=80)

ax.set_xlabel("Input variable 1")
ax.set_ylabel("Input variable 2")
ax.set_zlabel("Function output")
ax.set_title("3D View of Observed Function Values")

plt.show()

In [ ]:
# Ensure y is 1D
y = y.ravel()

if function == "function_x":
    y = -1 * y
    print(f'Inverted y:\n {y}')

n_features = X.shape[1]

# Maximum 3 charts per row
n_cols = min(3, n_features)
n_rows = math.ceil(n_features / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5 * n_cols, 4 * n_rows)
)

# Flatten axes array for easy indexing
axes = np.array(axes).reshape(-1)

best_idx = np.argmax(y)

for i in range(n_features):
    axes[i].scatter(X[:, i], y, alpha=0.7)
    axes[i].scatter(
        X[best_idx, i],
        y[best_idx],
        marker='*',
        s=250,
        label='Best observed'
    )
    
    axes[i].set_xlabel(f'x{i+1}')
    axes[i].set_ylabel('y')
    axes[i].set_title(f'y vs x{i+1}')
    axes[i].grid(True)
    axes[i].legend()

# Remove any unused subplots
for i in range(n_features, len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
plt.show()

# Function 1 Customization

In [ ]:
def find_epsilon_floor(y, top_fraction=0.5):
    """
    Auto-select a noise-floor epsilon by finding the largest gap in
    log10(|y|) among the upper portion of the data (likely-signal candidates).
    Restricting to the top fraction avoids being misled by sparse-sampling
    gaps deep in the noise tail, which don't represent a real noise/signal
    boundary — they're just an artifact of having few points there.
    """
    log_mag = np.log10(np.abs(y))
    sorted_log_mag = np.sort(log_mag)
    n = len(sorted_log_mag)
    n_top = max(2, int(np.ceil(n * top_fraction)))
    upper = sorted_log_mag[-n_top:]

    gaps = np.diff(upper)
    gap_idx = np.argmax(gaps)

    floor_log = (upper[gap_idx] + upper[gap_idx + 1]) / 2   # midpoint of the gap
    epsilon = 10 ** floor_log
    return epsilon, floor_log, gaps[gap_idx]

In [ ]:
#Function 1 Customisation

y_flipped = -y
print("Y Flipped", y_flipped)

epsilon, floor_log, gap_size = find_epsilon_floor(y, top_fraction=0.5)
print(f"Auto-selected epsilon: {epsilon:.3e}  "
      f"(floor at log10={floor_log:.2f}, gap size={gap_size:.2f} decades)")

#epsilon = np.min(y_flipped[y_flipped > 0]) #week 4
#y_pos = np.where(y_flipped > 0, y_flipped, epsilon)
#print(y_pos)
#y_log = np.log10(y_pos)
#print("Y log: ", y_log)

y_log = np.log10(np.abs(y) + epsilon)
print("Y log: ", y_log)

kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
         Matern(length_scale=0.3,              # single scalar, not a list
                length_scale_bounds=(0.05, 1.0),
                nu=2.5)
y_fit = y_log

df_y = pd.DataFrame({
    "X1": X[:,0],
    "X2": X[:,1],
    "y": y,
    "y_flipped": y_flipped,
    "y_log": y_log
})

output_file = f'./initial_data/{function}/y_transformations.csv'
df_y.to_csv(output_file, index=False)

In [ ]:
# Randomw grid sample that started for week 4 candidates

def random_grid_sample(n_grid=20, exclude_radius=0.1, X_observed=None, boundary_margin=0.05):
    # Create grid with margin inset from edges
    x1 = np.linspace(boundary_margin, 1 - boundary_margin, n_grid)
    x2 = np.linspace(boundary_margin, 1 - boundary_margin, n_grid)
    X_grid = np.array([[a, b] for a in x1 for b in x2])
    
    # Optionally exclude already-sampled regions
    if X_observed is not None:
        dists = np.min(
            np.linalg.norm(X_grid[:, None] - X_observed[None, :], axis=2), 
            axis=1
        )
        X_grid = X_grid[dists > exclude_radius]
    
    # Pick randomly from remaining grid points
    idx = np.random.randint(len(X_grid))
    return X_grid[idx]

print(X.shape)
next_point = random_grid_sample(n_grid=20, exclude_radius=0.1, X_observed=X, boundary_margin=0.05)
print("Next point: ", next_point)

In [ ]:
# --- Straddle / Level-Set acquisition ---
# Reframes "where to sample next" around a detection THRESHOLD T rather than
# the global best value, so it doesn't structurally favor whichever source is
# largest in magnitude (unlike EI/UCB).

#Week 5

def pick_threshold(floor_log, weakest_signal):
    """Formulaic T: midpoint between the noise floor and the weakest
    confirmed signal seen so far. Update weakest_signal each iteration."""
    return (floor_log + weakest_signal) / 2


def straddle_score(mu, sigma, T, z=1.96):
    """High when the model is unsure whether a point is above or below
    the detection threshold T (mu close to T, sigma large)."""
    return z * sigma - np.abs(mu - T)


def recommend_straddle_point(gpr, candidates, T, min_dist=0.10, X_existing=None):
    if min_dist is not None and X_existing is not None:
        d = np.linalg.norm(candidates[:, None, :] - X_existing[None, :, :], axis=2)
        candidates = candidates[d.min(axis=1) >= min_dist]

    mu, sigma = gpr.predict(candidates, return_std=True)
    straddle = straddle_score(mu, sigma, T)

    best_idx = np.argmax(straddle)
    return candidates[best_idx], mu[best_idx], sigma[best_idx], straddle[best_idx]

np.random.seed(42)
candidates = np.random.uniform(0.05, 0.95, size=(10000, X.shape[1]))  # boundary_margin=0.05

floor_log = -27.354         # from the gap-detection epsilon step
weakest_signal = -14.0155   # weakest confirmed real signal so far
T = pick_threshold(floor_log, weakest_signal)
print(f"Using T = {T:.3f} for straddle acquisition")

next_pt, mu_pt, sigma_pt, straddle_val = recommend_straddle_point(
    gpr, candidates, T, min_dist=0.10, X_existing=X
)

print(f"Next point to evaluate: {next_pt}")
print(f"Predicted mean: {mu_pt}")
print(f"Predicted std: {sigma_pt}")
print(f"Straddle score: {straddle_val}")

# Function 2 Customization

In [ ]:
kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
         RBF(length_scale=0.25, # single scalar — isotropic
             length_scale_bounds=(0.05, 2.0)) #wk3 & Wk5
             #length_scale_bounds=(0.25, 2.0)) #wk4

y_fit = y

df_y = pd.DataFrame({
    "X1": X[:,0],
    "X2": X[:,1],
    "y": y,
})

output_file = f'./initial_data/{function}/y_transformations.csv'
df_y.to_csv(output_file, index=False)

# Function 3 Customization

In [13]:
kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
         RBF(length_scale=0.3, # single scalar — isotropic
             length_scale_bounds=(0.2, 2.0))

y_fit = y

# Function 4 Customization

In [28]:
#kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
#         RBF(length_scale=0.3, # single scalar — isotropic
#             length_scale_bounds=(0.05, 2.0))

print("Create kernal for: ", function) #wk 5

kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(0.05, 2.0)
)
y_fit = y


Create kernal for:  function_4


# Function 5 Customization

In [37]:
# Wk3 remain unchanged
print("Create kernal for: ", function)
kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(1e-3, 1e3)
)
y_fit = y

Create kernal for:  function_5


# Function 6 Customization

In [48]:
# Wk3 remain unchanged
print("Create kernal for: ", function)
ARD = True

if ARD == False:
    kernel = ConstantKernel(1.0, constant_value_bounds=(0.01, 10.0)) * \
             RBF(length_scale=0.3, # single scalar — isotropic
                 length_scale_bounds=(0.05, 2.0))
else:
    #Wk 5
    kernel = 1.0 * RBF(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(0.05, 2.0)
    )
y_fit = y

Create kernal for:  function_6


# Function 7 Csutomization

In [56]:
# Wk3 remain unchanged
print("Create kernal for: ", function)
kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(0.05, 2.0)
)
y_fit = y

Create kernal for:  function_7


# Function 8 Customization

In [64]:
# Wk3 remain unchanged
#wk4 change the caps
print("Create kernal for: ", function)

kernel = 1.0 * RBF(
    length_scale=np.ones(X.shape[1]),
    length_scale_bounds=(0.05, 20.0)
)
y_fit = y

Create kernal for:  function_8


# Fit Gaussian Process

In [65]:
# Fit Gaussian Process

gpr = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)
gpr.fit(X, y_fit)
print(gpr.kernel_)

print(f"LML:       {gpr.log_marginal_likelihood_value_:.4f}")

2.7**2 * RBF(length_scale=[1.5, 2.12, 0.992, 5.56, 10.9, 2.01, 1.5, 20])
LML:       1.5677


C:\Users\scbar\anaconda3\envs\scipy-clean\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 20.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [66]:
# Generate candidates and predict y

if X.shape[1] <= 2:
    # Create grid of candidate points
    x1 = np.linspace(0, 1, 100)
    x2 = np.linspace(0, 1, 100)

    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])

    print("Shape:", X1.shape)

    print("\nTop-left 5x5 block:")
    print(X1[:5, :5])

    print("\nX1 ravel:")
    print(X1.ravel()[:5])

    print("\nGrid Type:", type(grid))
    print("Shape:", grid.shape)
    print("Dimensions:", grid.ndim)
    print("Data type:", grid.dtype)
    print("\nFirst 5 rows:")
    print(grid[:5])
else:
    print("Grid is greater than 3 dimensions")
    n_candidates = 10000
    low = np.zeros(X.shape[1])
    high = np.ones(X.shape[1])
    
    if function == "function_3":
        high[2] = 0.5   # constrain x3 (3rd column, index 2) to [0, 0.6)
    
    grid = np.random.uniform(
    low=low,
    high=high,
    size=(n_candidates, X.shape[1]))
    
    print("Shape:", grid.shape)
    print("Dimensions:", grid.ndim)
    print("\nFirst 5 rows:")
    print(grid[:5])
    
# GP predictions
mu, sigma = gpr.predict(grid, return_std=True)


Grid is greater than 3 dimensions
Shape: (10000, 8)
Dimensions: 2

First 5 rows:
[[0.58116694 0.28838177 0.24178966 0.01244421 0.62792277 0.85715198
  0.29703636 0.85683079]
 [0.588399   0.69936242 0.7556266  0.81805898 0.22909924 0.28118067
  0.70739798 0.15568492]
 [0.93478153 0.42132164 0.51342248 0.23852042 0.08112407 0.27939051
  0.89074563 0.63868608]
 [0.48841467 0.03995027 0.22835913 0.18666816 0.17268079 0.91455196
  0.64963667 0.84024945]
 [0.12603779 0.0312773  0.54975516 0.36319179 0.0234933  0.51541118
  0.22658975 0.91006413]]


# Define Acquisition Function

In [67]:
 
xi_values = {
    "function_1": 0.5, #RBF to Matern wk3
    "function_2": 0.2, # 0.05 to 0.5 for wk4
    "function_3": 0.01, #0.01 to 0.1 wk4
    "function_4": 0.01, #0.1 to 0.01 wk3
    "function_5": 0.01,
    "function_6": 0.05,
    "function_7": 0.05, #0.05 to 0.3 wk3
    "function_8": 0.1
}

In [68]:
# Expected Improvement acquisition function
y_best = np.max(y_fit)


xi = xi_values.get(function, 0.1)

print(f"Using xi = {xi} for {function}")

improvement = mu - y_best - xi
Z = improvement / sigma

ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0.0] = 0.0

# Pick next point
next_idx = np.argmax(ei)
print("Next Idx: ", next_idx)
next_x = grid[next_idx]

print("Current best y:", y_best)
print("Next point to evaluate:", next_x)
print("Formatted next point to evaluate:", "-".join(f"{x:.6f}" for x in next_x))
print("Expected improvement:", ei[next_idx])

print("Predicted mean:", mu[next_idx])
print("Predicted std:", sigma[next_idx])
print("Improvement:", mu[next_idx] - y_best - xi)

Using xi = 0.1 for function_8
Next Idx:  8980
Current best y: 9.9264226345365
Next point to evaluate: [0.05922576 0.18115649 0.00109351 0.27716746 0.78097659 0.86436064
 0.08177517 0.89117689]
Formatted next point to evaluate: 0.059226-0.181156-0.001094-0.277167-0.780977-0.864361-0.081775-0.891177
Expected improvement: 0.00039179618372853523
Predicted mean: 9.749576838005671
Predicted std: 0.11856983118833253
Improvement: -0.27684579653082897


In [33]:
# UCB acquisition function

# ---- ACQUISITION: GP-UCB (replaces EI) ----
kappa = 10
ucb = mu + kappa * sigma
 
next_idx = np.argmax(ucb)
 
print(f"\nUsing kappa = {kappa} for function_4")
print("Next Idx: ", next_idx)
print("Current best y:", y_best)
print("Next point to evaluate:", grid[next_idx])
print("UCB value:", ucb[next_idx])
print("Predicted mean:", mu[next_idx])
print("Predicted std:", sigma[next_idx])


Using kappa = 10 for function_4
Next Idx:  963
Current best y: -0.04243494859040764
Next point to evaluate: [0.00637904 0.97835479 0.96574721 0.3951553 ]
UCB value: 39.811501457673224
Predicted mean: -23.833995248079965
Predicted std: 6.364549670575319


In [ ]:
# Function 2 max variance
import sys
sys.path.append('./Utils')

from imse_acquisition import imse_next_points

# after your usual GP fit:
gpr.fit(X, y_fit)

points, scores = imse_next_points(
    X, y,
    kernel=gpr.kernel_,          # use the OPTIMIZED kernel, not an unfitted one
    n_points=2,                 # how many coverage points you want this round
    alpha=1e-6,                 # match your GaussianProcessRegressor alpha
    boundary_margin=0.05,
    domain=(0.0, 1.0),
    normalize_y=True,
)
print(points)   # the two next points to physically evaluate

#Point: [0.730, 0.546]   IMSE score: 0.02315   (Region 2)
#Point: [0.398, 0.912]   IMSE score: 0.02285   (near Region 1)

In [24]:
#Function 3 customise
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]
r = 0.12

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(3)])
bounds_high = np.array([min(0.55 if i == 2 else 0.95, x_best[i] + r) for i in range(3)])

sampler = LatinHypercube(d=3, seed=42)
lhs = sampler.random(n=20)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

# Pick candidate with highest GP mean
mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

Next point: [0.26119165 0.31480591 0.4619131 ]
Formatted: 0.261192-0.314806-0.461913
mu: -0.0058129556735053295


In [34]:
#Function 4 customise

from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P31
r = 0.08  # tight radius — basin is narrow

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(4)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(4)])

sampler = LatinHypercube(d=4, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

Next point: [0.37264484 0.47927348 0.46173147 0.41305467]
Formatted: 0.372645-0.479273-0.461731-0.413055
mu: 0.2839958657841386


In [53]:
#Fuction 6 customise
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P24
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(5)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(5)])

print("Search bounds:", list(zip(bounds_low, bounds_high)))

sampler = LatinHypercube(d=5, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

Search bounds: [(np.float64(0.33300699999999994), np.float64(0.533007)), (np.float64(0.06703799999999999), np.float64(0.267038)), (np.float64(0.45945199999999997), np.float64(0.6594519999999999)), (np.float64(0.56132), np.float64(0.76132)), (np.float64(0.05), np.float64(0.139541))]
Next point: [0.51778956 0.10916847 0.64831295 0.67324865 0.13397308]
Formatted: 0.517790-0.109168-0.648313-0.673249-0.133973
mu: -0.4454922349349213


In [61]:
#Customise for Function 7
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P34
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(6)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(6)])

# Override x3 ceiling
bounds_high[2] = 0.30

sampler = LatinHypercube(d=6, seed=42)
lhs = sampler.random(n=50)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

Next point: [0.1145916  0.15290264 0.13465791 0.18030894 0.32698472 0.86545189]
Formatted: 0.114592-0.152903-0.134658-0.180309-0.326985-0.865452
mu: 1.9506758507654252


In [70]:
#Customise function 8
from scipy.stats.qmc import LatinHypercube

x_best = X[np.argmax(y)]  # P44 [0.105, 0.077, 0.174, 0.049, 0.906, 0.632, 0.308, 0.362]
r = 0.10

bounds_low  = np.array([max(0.05, x_best[i] - r) for i in range(8)])
bounds_high = np.array([min(0.95, x_best[i] + r) for i in range(8)])

# Key constraints
bounds_high[0] = 0.12   # x1 ceiling
bounds_high[2] = 0.15   # x3 ceiling
bounds_low[4]  = 0.90   # x5 floor
bounds_high[6] = 0.25   # x7 ceiling

print("Bounds:", list(zip(bounds_low.round(3), bounds_high.round(3))))

sampler = LatinHypercube(d=8, seed=42)
lhs = sampler.random(n=100)
candidates = bounds_low + lhs * (bounds_high - bounds_low)

mu_c, _ = gpr.predict(candidates, return_std=True)
next_x = candidates[np.argmax(mu_c)]
print("Next point:", next_x)
print("Formatted:", "-".join(f"{x:.6f}" for x in next_x))
print("mu:", mu_c[np.argmax(mu_c)])

Bounds: [(np.float64(0.05), np.float64(0.12)), (np.float64(0.05), np.float64(0.177)), (np.float64(0.074), np.float64(0.15)), (np.float64(0.05), np.float64(0.149)), (np.float64(0.9), np.float64(0.95)), (np.float64(0.532), np.float64(0.732)), (np.float64(0.208), np.float64(0.25)), (np.float64(0.262), np.float64(0.462))]
Next point: [0.09473343 0.15617305 0.12791644 0.08658037 0.9449313  0.61212924
 0.21594956 0.30282264]
Formatted: 0.094733-0.156173-0.127916-0.086580-0.944931-0.612129-0.215950-0.302823
mu: 9.949824940858353


# Function 1 Plots for Uncertainty and Signal Strength 

In [ ]:
# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Uncertainty
im1 = axes[0].contourf(x1g, x2g, sigma, levels=50, cmap='YlOrRd')
axes[0].scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=60, zorder=5, label='Evaluated')
plt.colorbar(im1, ax=axes[0], label='GP std (σ)')
axes[0].set_title('Unexplored Regions (GP Uncertainty)', fontsize=13)
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].legend(loc='upper left', fontsize=9)

# Panel 2: Mean signal (diverging — negative = source regions)
vmax = np.abs(mu).max()
im2 = axes[1].contourf(x1g, x2g, mu, levels=50, cmap='RdBu', vmin=-vmax, vmax=vmax)
axes[1].scatter(X[:, 0], X[:, 1], c='black', edgecolors='white', s=60, zorder=5, label='Evaluated')
# Annotate strong/weak sources
axes[1].annotate('Strong\nsource', xy=(0.636, 0.677), fontsize=8, color='white',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))
axes[1].annotate('Weak\nsource', xy=(0.313, 0.333), fontsize=8, color='white',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc='navy', alpha=0.5))
axes[1].annotate('Weak\nsource', xy=(0.731, 0.733), fontsize=8, color='white',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc='navy', alpha=0.5))
plt.colorbar(im2, ax=axes[1], label='GP mean (μ, transformed)')
axes[1].set_title('Signal Strength (GP Mean)', fontsize=13)
axes[1].set_xlabel('x₁'); axes[1].set_ylabel('x₂')
axes[1].legend(loc='upper left', fontsize=9)

plt.suptitle('Function 1 — Contamination Source Detection (n=15)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('./Figures/Function1_exploration_signal.png', dpi=150, bbox_inches='tight')
plt.show()
